In [12]:
%pip install torch torchvision tqdm matplotlib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
from tqdm import tqdm
import matplotlib.pyplot as plt


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
BATCH_SIZE = 64
IMG_SIZE = 224

transform_train = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])
transform_test = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

trainset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)

train_loader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

classes = trainset.classes

In [14]:
class FFTBranch(nn.Module):
    def __init__(self, in_channels=3, out_dim=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(64, out_dim)

    def forward(self, x):
        # Compute log-magnitude FFT
        B, C, H, W = x.shape
        fft = torch.fft.fft2(x)
        fft_mag = torch.log1p(torch.abs(fft))
        feats = self.conv(fft_mag).flatten(1)
        return self.fc(feats)

In [15]:
import torchvision.models as models

class HybridFFTModel(nn.Module):
    def __init__(self, backbone="resnet50", pretrained=True, fft_out=256, num_classes=10):
        super().__init__()
        if backbone == "resnet50":
            net = models.resnet50(pretrained=pretrained)
            feat_dim = net.fc.in_features
            self.backbone = nn.Sequential(*list(net.children())[:-1])  # remove FC
        else:
            raise NotImplementedError("Only ResNet50 implemented for now")

        self.fft_branch = FFTBranch(in_channels=3, out_dim=fft_out)
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim + fft_out, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        bfeat = self.backbone(x).flatten(1)
        ffeat = self.fft_branch(x)
        fused = torch.cat([bfeat, ffeat], dim=1)
        return self.classifier(fused)

In [16]:
def train_one_epoch(model, loader, opt, device, criterion):
    model.train()
    total, correct, loss_sum = 0, 0, 0
    for x, y in tqdm(loader, leave=False):
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        opt.step()
        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, correct / total

def evaluate(model, loader, device, criterion):
    model.eval()
    total, correct, loss_sum = 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            loss_sum += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return loss_sum / total, correct / total

In [17]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HybridFFTModel(backbone="resnet50", pretrained=True).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-4},
    {'params': model.fft_branch.parameters(), 'lr': 1e-3},
    {'params': model.classifier.parameters(), 'lr': 1e-3}
])

EPOCHS = 5
train_losses, val_losses, val_accs = [], [], []

for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, device, criterion)
    val_loss, val_acc = evaluate(model, test_loader, device, criterion)
    train_losses.append(tr_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss {tr_loss:.3f} | Val Acc {val_acc:.3f}")

KeyboardInterrupt: 

In [ ]:
# 1️⃣ Backbone-only model
class BackboneOnly(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        net = models.resnet50(pretrained=pretrained)
        self.backbone = nn.Sequential(*list(net.children())[:-1])
        self.fc = nn.Linear(net.fc.in_features, 10)
    def forward(self, x):
        feat = self.backbone(x).flatten(1)
        return self.fc(feat)

# 2️⃣ FFT-only model
class FFTOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.fft = FFTBranch(out_dim=256)
        self.fc = nn.Linear(256, 10)
    def forward(self, x):
        return self.fc(self.fft(x))

def quick_train(model, name, epochs=3):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    for e in range(epochs):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, opt, device, criterion)
        val_loss, val_acc = evaluate(model, test_loader, device, criterion)
        print(f"{name} Epoch {e+1}/{epochs} | Val Acc: {val_acc:.3f}")
    return val_acc

acc_backbone = quick_train(BackboneOnly(), "Backbone-only")
acc_fft = quick_train(FFTOnly(), "FFT-only")
print(f"Hybrid model achieved {val_accs[-1]:.3f} vs Backbone {acc_backbone:.3f} vs FFT {acc_fft:.3f}")

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(val_accs, label='Hybrid FFT+Backbone')
plt.axhline(y=acc_backbone, color='r', linestyle='--', label='Backbone-only')
plt.axhline(y=acc_fft, color='g', linestyle='--', label='FFT-only')
plt.title('Validation Accuracy Comparison')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()